# テニス試合動画の解析（ボールの軌跡・バウンド・イン/アウト・サーブ速度）

**使い方**: 上のメニュー「ランタイム」→「ランタイムのタイプを変更」で **T4 GPU** を選び、「ランタイム」→「**すべてのセルを実行**」。
最後に Google ドライブの許可を求められたら許可すると、`マイドライブ/テニス解析/` に完成した動画が保存されます。

25 分の試合で 20〜40 分ほどかかります（多くは動画のダウンロードとボール検出）。

In [ ]:
#@title 設定（IMG_0131 用。別の動画では CAM の位置合わせが必要）
VIDEO_ID = "1R-U6YRT-jXp_naOH12etyvpWDXCtChTX"   # Google ドライブの動画（リンクを知っている全員が閲覧可）
START_SEC = 190                                  # これより前（サーブ練習）は使わない
CAM = "data/IMG_0131_cam.npz"                   # コートの位置合わせ結果
OUT_NAME = "IMG_0131_解析.mp4"

In [ ]:
!nvidia-smi -L || echo "GPU がありません（ランタイムのタイプで T4 GPU を選んでください）"
!apt-get -qq install -y fonts-ipafont-gothic > /dev/null
!pip -q install gdown
!rm -rf repo && git clone -q --depth 1 https://github.com/air-tomo/air-tomo.github.io repo
%cd repo/golf-swing-app/tools/court3d

In [ ]:
# 動画（約 2.9GB）と学習済みのボール検出モデル（TrackNet）をダウンロード
!gdown -q "https://drive.google.com/uc?id={VIDEO_ID}" -O /content/video.mov
!gdown -q "https://drive.google.com/uc?id=1XEYZ4myUN7QT-NeBYJI0xteLsvs-ZAOl" -O /content/model_best.pt
!ls -la /content/video.mov /content/model_best.pt

In [ ]:
# 解析と動画づくり（ポイントごとにボール検出 → バウンド・イン/アウト・サーブ速度 → 重ねた動画）
!python3 make_video.py /content/video.mov {CAM} /content/model_best.pt /content/out --start {START_SEC}

In [ ]:
# Google ドライブに保存
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
os.makedirs('/content/drive/MyDrive/テニス解析', exist_ok=True)
shutil.copy('/content/out/final.mp4', '/content/drive/MyDrive/テニス解析/' + OUT_NAME)
shutil.copy('/content/out/tn_post.json', '/content/drive/MyDrive/テニス解析/' + OUT_NAME.replace('.mp4', '.json'))
print('保存しました: マイドライブ/テニス解析/' + OUT_NAME)